# Notebook 03 — CLIP Feature Labeling
Loads trained SAE checkpoints and labels each feature using CLIP.
Processes a subset of features (first 500 per layer) for speed.

Output: `../checkpoints/labels_layer4.pt`, `../checkpoints/labels_layer12.pt`
Each is a dict: {'labels': [...], 'scores': [...], 'top_k_indices': [...]}

In [1]:
import torch
print(torch.__version__)

2.7.0+cu126


In [2]:
import sys
sys.path.insert(0, '..')
import torch
from torchvision import datasets, transforms
from PIL import Image
from src.sae import SAE
from src.clip_labeler import label_features, compute_correlation_scores

DEVICE             = 'cuda'
IMAGENET_VAL_DIR   = 'D:/Master Material/XAI/XAI_project/archive/imagenet-val'
N_IMAGES           = 10000
N_FEATURES         = 500
K_PATCHES          = 10
N_SAMPLE_TOKENS    = 50000


In [3]:
pil_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
dataset = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=None)

# Load the indices saved during activation collection so each activation
# position maps to the correct original image, not img 0..N sequentially.
selected_indices = torch.load('../data/selected_indices.pt', weights_only=True)
n_to_load = min(N_IMAGES, len(selected_indices))

token_to_image = {}
for pos, orig_idx in enumerate(selected_indices[:n_to_load]):
    img_path = dataset.imgs[int(orig_idx)][0]
    img = Image.open(img_path).convert('RGB')
    img = pil_transform(img)
    for j in range(196):  # patch tokens only, CLS excluded
        token_to_image[pos * 196 + j] = (img, j + 1)  # tok_offset 1-196

print(f'token_to_image entries: {len(token_to_image)} '
      f'(expect {n_to_load * 196})')

token_to_image entries: 1960000 (expect 1960000)


In [4]:
def label_layer(acts_path, ckpt_path, save_path, layer_name):
    acts = torch.load(acts_path, weights_only=False)

    patch_mask = torch.arange(acts.shape[0]) % 197 != 0
    acts = acts[patch_mask]

    idx = torch.randperm(acts.shape[0], generator=torch.Generator().manual_seed(42))[:N_SAMPLE_TOKENS]
    acts_sample = acts[idx]

    ckpt = torch.load(ckpt_path, weights_only=False)
    sae = SAE(d_input=768, d_hidden=3072).cpu()
    sae.load_state_dict(ckpt['state_dict'])
    sae.eval()

    acts_norm = (acts_sample - ckpt['acts_mean']) / ckpt['acts_rms']

    CHUNK = 4096
    sae_acts_list = []
    with torch.no_grad():
        for start in range(0, acts_norm.shape[0], CHUNK):
            f, _ = sae(acts_norm[start:start+CHUNK])
            sae_acts_list.append(f)
    sae_acts = torch.cat(sae_acts_list, dim=0)

    t2i_sampled = {new_i: token_to_image[idx[new_i].item()]
                   for new_i in range(len(idx)) if idx[new_i].item() in token_to_image}

    print(f'CLIP labeling {layer_name} ({N_FEATURES} features)...')
    results = label_features(sae_acts[:, :N_FEATURES], t2i_sampled,
                             k=K_PATCHES, device=DEVICE)

    print(f'Computing top-vs-bottom separation scores for {layer_name}...')
    results['corr_scores'] = compute_correlation_scores(
        sae_acts[:, :N_FEATURES], t2i_sampled, results['labels'], device=DEVICE)
    results['sep_scores'] = results['corr_scores']

    torch.save(results, save_path)
    print(f'Saved {save_path}')
    from collections import Counter
    top5 = Counter(results['labels']).most_common(5)
    print(f'Top-5 labels: {top5}')
    print(f'Mean cosine score: {sum(results["scores"])/len(results["scores"]):.3f}')
    print(f'Mean separation score: {sum(results["sep_scores"])/len(results["sep_scores"]):.3f}')
    return results


In [5]:
labels4 = label_layer(
    '../data/layer4_activations.pt',
    '../checkpoints/sae_layer4.pt',
    '../checkpoints/labels_layer4.pt',
    'layer4',
)

CLIP labeling layer4 (500 features)...


d:\Tools\Anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Computing top-vs-bottom separation scores for layer4...
Saved ../checkpoints/labels_layer4.pt
Top-5 labels: [('edge', 62), ('slot', 31), ('muzzle', 31), ('web site', 29), ('patas', 27)]
Mean cosine score: 0.287
Mean separation score: 0.023


In [6]:
labels8 = label_layer(
    '../data/layer8_activations.pt',
    '../checkpoints/sae_layer8.pt',
    '../checkpoints/labels_layer8.pt',
    'layer8',
)

CLIP labeling layer8 (500 features)...
Computing top-vs-bottom separation scores for layer8...
Saved ../checkpoints/labels_layer8.pt
Top-5 labels: [('edge', 33), ('muzzle', 28), ('slot', 21), ('background', 18), ('patas', 18)]
Mean cosine score: 0.290
Mean separation score: 0.070


In [7]:
labels12 = label_layer(
    '../data/layer12_activations.pt',
    '../checkpoints/sae_layer12.pt',
    '../checkpoints/labels_layer12.pt',
    'layer12',
)

CLIP labeling layer12 (500 features)...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 97c0b2b5-5cb3-4cff-b73c-c7aab6f41a69)')' thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Computing top-vs-bottom separation scores for layer12...
Saved ../checkpoints/labels_layer12.pt
Top-5 labels: [('muzzle', 39), ('edge', 36), ('slot', 23), ('patas', 21), ('background', 15)]
Mean cosine score: 0.290
Mean separation score: 0.080


In [8]:
def label_raw_neurons(acts_path, save_path, layer_name, n_neurons=100):
    """Label raw ViT activation dimensions (no SAE) for baseline comparison."""
    acts = torch.load(acts_path, weights_only=False)
    patch_mask = torch.arange(acts.shape[0]) % 197 != 0
    acts = acts[patch_mask]

    idx = torch.randperm(acts.shape[0], generator=torch.Generator().manual_seed(99))[:N_SAMPLE_TOKENS]
    acts_sample = acts[idx]  # raw ViT activations, no SAE

    t2i_sampled = {new_i: token_to_image[idx[new_i].item()]
                   for new_i in range(len(idx)) if idx[new_i].item() in token_to_image}

    print(f'CLIP labeling raw neurons in {layer_name} ({n_neurons} dims)...')
    results = label_features(acts_sample[:, :n_neurons], t2i_sampled,
                             k=K_PATCHES, device=DEVICE)
    results['corr_scores'] = compute_correlation_scores(
        acts_sample[:, :n_neurons], t2i_sampled, results['labels'], device=DEVICE)
    results['sep_scores'] = results['corr_scores']

    torch.save(results, save_path)
    print(f'Saved {save_path}')
    print(f'Mean cosine score: {sum(results["scores"])/len(results["scores"]):.3f}')
    print(f'Mean separation score: {sum(results["sep_scores"])/len(results["sep_scores"]):.3f}')
    return results

raw4  = label_raw_neurons('../data/layer4_activations.pt',
                          '../checkpoints/raw_labels_layer4.pt',  'layer4')
raw8  = label_raw_neurons('../data/layer8_activations.pt',
                          '../checkpoints/raw_labels_layer8.pt',  'layer8')
raw12 = label_raw_neurons('../data/layer12_activations.pt',
                          '../checkpoints/raw_labels_layer12.pt', 'layer12')

CLIP labeling raw neurons in layer4 (100 dims)...
Saved ../checkpoints/raw_labels_layer4.pt
Mean cosine score: 0.286
Mean separation score: 0.091
CLIP labeling raw neurons in layer8 (100 dims)...
Saved ../checkpoints/raw_labels_layer8.pt
Mean cosine score: 0.286
Mean separation score: 0.079
CLIP labeling raw neurons in layer12 (100 dims)...
Saved ../checkpoints/raw_labels_layer12.pt
Mean cosine score: 0.286
Mean separation score: 0.081
